In [ ]:
import os
import cv2
import glob
import shutil
import numpy as np
from tqdm import tqdm
import albumentations as A
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR = "/home/kartik/Desktop/shalini/brain_brisc/brisc/segmentation_task"
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train", "images")
TRAIN_MASK_DIR = os.path.join(DATA_DIR, "train", "masks")
OUTPUT_DIR = "/home/kartik/Desktop/shalini/brain_brisc/brisc_processed"
OUT_TRAIN_IMG = os.path.join(OUTPUT_DIR, "train", "images")
OUT_TRAIN_MASK = os.path.join(OUTPUT_DIR, "train", "masks")
OUT_VAL_IMG = os.path.join(OUTPUT_DIR, "val", "images")
OUT_VAL_MASK = os.path.join(OUTPUT_DIR, "val", "masks")
for d in [OUT_TRAIN_IMG, OUT_TRAIN_MASK, OUT_VAL_IMG, OUT_VAL_MASK]:
    os.makedirs(d, exist_ok=True)

In [ ]:
image_paths = sorted(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg")))
data_info = []
for p in image_paths:
    basename = os.path.basename(p)
    parts = basename.replace(".jpg", "").split("_")
    tumor_type = parts[3]
    plane = parts[4]
    mask_name = basename.replace(".jpg", ".png")
    mask_path = os.path.join(TRAIN_MASK_DIR, mask_name)
    stratify_key = f"{tumor_type}_{plane}"
    data_info.append({
        "image_path": p,
        "mask_path": mask_path,
        "basename": basename,
        "tumor_type": tumor_type,
        "plane": plane,
        "stratify_key": stratify_key
    })
print(f"Total training samples: {len(data_info)}")
print("\nComputing pixel-level foreground/background ratio per tumor type...")
tumor_stats = {}
for info in tqdm(data_info, desc="Analyzing masks"):
    mask = cv2.imread(info["mask_path"], cv2.IMREAD_GRAYSCALE)
    if mask is None: continue
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    fg_pixels = np.sum(binary_mask == 255)
    bg_pixels = np.sum(binary_mask == 0)
    tt = info["tumor_type"]
    if tt not in tumor_stats:
        tumor_stats[tt] = {"fg": 0, "bg": 0}
    tumor_stats[tt]["fg"] += fg_pixels
    tumor_stats[tt]["bg"] += bg_pixels
for tt, stats in tumor_stats.items():
    ratio = stats["fg"] / (stats["bg"] + 1e-8)
    print(f"Tumor Type: {tt.upper()} | FG:BG Ratio = {ratio:.4f} | FG Pixels: {stats['fg']} | BG Pixels: {stats['bg']}")

In [ ]:
X = data_info
y = [info["stratify_key"] for info in data_info]
train_info, val_info = train_test_split(X, test_size=0.20, random_state=42, stratify=y)
print(f"Train split: {len(train_info)} samples")
print(f"Val split: {len(val_info)} samples")

In [ ]:
def preprocess_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read {img_path}")
    img_float = img.astype(np.float32)
    mean = np.mean(img_float)
    std = np.std(img_float)
    if std > 0:
        img_z = (img_float - mean) / std
    else:
        img_z = img_float - mean
    min_val = np.min(img_z)
    max_val = np.max(img_z)
    if max_val - min_val > 0:
        img_norm = (img_z - min_val) / (max_val - min_val)
    else:
        img_norm = img_z
    img_uint8 = (img_norm * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_clahe = clahe.apply(img_uint8)
    return img_clahe
def preprocess_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Could not read {mask_path}")
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    return binary_mask
augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5)
])

In [ ]:
def process_and_save_dataset(dataset_info, split_type, num_augmentations=1):
    is_train = (split_type == 'train')
    out_img_dir = OUT_TRAIN_IMG if is_train else OUT_VAL_IMG
    out_mask_dir = OUT_TRAIN_MASK if is_train else OUT_VAL_MASK
    for info in tqdm(dataset_info, desc=f"Processing {split_type} set"):
        basename = info["basename"]
        mask_name = basename.replace(".jpg", ".png")
        img_proc = preprocess_image(info["image_path"])
        mask_proc = preprocess_mask(info["mask_path"])
        cv2.imwrite(os.path.join(out_img_dir, basename), img_proc)
        cv2.imwrite(os.path.join(out_mask_dir, mask_name), mask_proc)
        if is_train:
            for i in range(num_augmentations):
                augmented = augmentation_pipeline(image=img_proc, mask=mask_proc)
                aug_img = augmented['image']
                aug_mask = augmented['mask']
                aug_basename = basename.replace(".jpg", f"_aug{i}.jpg")
                aug_mask_name = mask_name.replace(".png", f"_aug{i}.png")
                cv2.imwrite(os.path.join(out_img_dir, aug_basename), aug_img)
                cv2.imwrite(os.path.join(out_mask_dir, aug_mask_name), aug_mask)
print("Processing Validation Set (No Augmentations)...")
process_and_save_dataset(val_info, split_type='val', num_augmentations=0)
print("\nProcessing Training Set (With Augmentations)...")
process_and_save_dataset(train_info, split_type='train', num_augmentations=1)
print("\nDataset processing complete!")